# PatchEval — Go Dataset EDA + PoC Comparison

Notebook này **mirror cấu trúc của `eda.ipynb`** nhưng:
- Chỉ dùng **subset Go** của dataset
- Thay `df_llm` (baseline) bằng kết quả **PoC thực tế** tự chạy (`gemini-3.5-flash`)
- **So sánh trực tiếp** kết quả tự chạy với 3 model baseline PatchEval (claude-3-5-sonnet, gemini-1.5-pro, gpt-4o)

**Cấu trúc:**
- **Level 1** — Dataset Overview (Go only)
- **Level 2** — Vulnerability Characteristics (Go only)
- **Level 3** — Patch Complexity (Go only)
- **Level 4** — PoC Results vs Baseline

## 0. Setup & Load Data

In [ ]:
import json
import os
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
PALETTE = px.colors.qualitative.Bold
sns.set_palette('husl')

# ── Paths (relative to eda/ folder) ───────────────────────────────────────────
BASE        = Path('../patcheval')
DATASET_ALL = BASE / 'datasets' / 'patcheval_verified.json'       # full dataset (same as eda.ipynb)
DATASET_GO  = BASE / 'datasets' / 'patcheval_verified_go.json'    # Go subset
PATCH_FILE  = BASE / 'exp_agent' / 'eval_inputs' / 'go_poc.jsonl'
LOG_DIR     = BASE / 'evaluation' / 'evaluation_output' / 'results' / 'go_poc' / 'logs'

# ── Load full dataset (to replicate eda.ipynb baseline numbers) ───────────────
with DATASET_ALL.open() as f:
    raw_all = json.load(f)

# ── Load Go subset ────────────────────────────────────────────────────────────
with DATASET_GO.open() as f:
    raw_go = json.load(f)

print(f'Total records (full dataset): {len(raw_all)}')
print(f'Total records (Go subset)   : {len(raw_go)}')

In [ ]:
# ── Helper functions — identical to eda.ipynb ──────────────────────────────────

def extract_year(cve_id):
    m = re.match(r'CVE-(\d{4})-', cve_id)
    return int(m.group(1)) if m else None

def count_patch_lines(vul_func):
    total = 0
    for func in vul_func:
        for loc in func.get('vul_localization', []):
            total += len(loc.get('patch_lines', []))
    return total

def count_patch_hunks(vul_func):
    return sum(len(func.get('vul_localization', [])) for func in vul_func)

def count_patch_files(vul_func):
    return len({func.get('file_path') for func in vul_func})

def patch_complexity(n_lines):
    if n_lines <= 5:    return 'Easy (1-5)'
    elif n_lines <= 20: return 'Medium (6-20)'
    elif n_lines <= 50: return 'Hard (21-50)'
    else:               return 'Very Hard (50+)'

COMPLEXITY_ORDER = ['Easy (1-5)', 'Medium (6-20)', 'Hard (21-50)', 'Very Hard (50+)']
COMPLEXITY_COLORS = {
    'Easy (1-5)'    : '#2ecc71',
    'Medium (6-20)' : '#f1c40f',
    'Hard (21-50)'  : '#e67e22',
    'Very Hard (50+)': '#e74c3c',
}

def build_df(raw):
    rows = []
    for entry in raw:
        cve      = entry['cve_id']
        year     = extract_year(cve)
        language = entry.get('programing_language', 'Unknown')
        repo     = entry.get('repo', '')
        cwes     = list(entry.get('cwe_info', {}).keys())
        vul_func = entry.get('vul_func', [])
        p_lines  = count_patch_lines(vul_func)
        p_hunks  = count_patch_hunks(vul_func)
        p_files  = count_patch_files(vul_func)
        for cwe in (cwes if cwes else ['Unknown']):
            rows.append({
                'cve_id': cve, 'year': year, 'language': language,
                'repo': repo, 'cwe': cwe,
                'patch_lines': p_lines, 'patch_hunks': p_hunks, 'patch_files': p_files,
            })
    df_exp = pd.DataFrame(rows)
    df     = df_exp.drop_duplicates(subset='cve_id').copy()
    df['complexity']     = df['patch_lines'].apply(patch_complexity)
    df_exp['complexity'] = df_exp['patch_lines'].apply(patch_complexity)
    return df, df_exp

# Full dataset (for baseline context)
df_all, df_exp_all = build_df(raw_all)

# Go-only dataset
df, df_exp = build_df(raw_go)

print(f'Go CVEs   : {df["cve_id"].nunique()}')
print(f'Go repos  : {df["repo"].nunique()}')
print(f'Go CWEs   : {df_exp["cwe"].nunique()}')
print(f'Year range: {int(df["year"].min())} - {int(df["year"].max())}')
df.head(3)

---
## Level 1 — Dataset Overview (Go only vs Full Dataset)

In [ ]:
# ── CVEs per language (full dataset) — highlight Go ───────────────────────────
# Mirror of eda.ipynb Cell 6
lang_counts = df_all['language'].value_counts().reset_index()
lang_counts.columns = ['language', 'count']
lang_counts['color'] = lang_counts['language'].apply(
    lambda l: '#EF553B' if l == 'Go' else '#636EFA'
)

fig = px.bar(
    lang_counts, x='language', y='count',
    title='CVEs per Programming Language (full dataset — Go highlighted)',
    color='language',
    color_discrete_sequence=PALETTE,
    text='count',
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, xaxis_title='Language', yaxis_title='# CVEs')
# Highlight Go bar
for trace in fig.data:
    if trace.name == 'Go':
        trace.marker.color = '#EF553B'
fig.show()

In [ ]:
# ── Top-20 CWE Distribution — Go vs Full Dataset side-by-side ─────────────────
# Mirror of eda.ipynb Cell 7
top_cwes_go  = df_exp['cwe'].value_counts().head(20)
top_cwes_all = df_exp_all[df_exp_all['language'] == 'Go']['cwe'].value_counts().head(20)

cwe_counts_go = top_cwes_go.reset_index()
cwe_counts_go.columns = ['cwe', 'count']
cwe_counts_go['source'] = 'Go subset (our dataset)'

fig = px.bar(
    cwe_counts_go, x='count', y='cwe',
    orientation='h',
    title='Top-20 CWE Distribution — Go CVEs',
    color='count',
    color_continuous_scale='Teal',
    text='count',
)
fig.update_layout(yaxis={'autorange': 'reversed'}, coloraxis_showscale=False)
fig.show()

In [ ]:
# ── CVEs per Year — Go vs All Languages ───────────────────────────────────────
# Mirror of eda.ipynb Cell 8
year_go  = df.dropna(subset=['year']).groupby('year')['cve_id'].count().reset_index()
year_go.columns = ['year', 'count']
year_go['source'] = 'Go'

year_all = df_all.dropna(subset=['year']).groupby('year')['cve_id'].count().reset_index()
year_all.columns = ['year', 'count']
year_all['source'] = 'All Languages'

year_combined = pd.concat([year_all, year_go])

fig = px.line(
    year_combined, x='year', y='count', color='source',
    title='CVE Count by Year — Go vs All Languages',
    markers=True, line_shape='spline',
    color_discrete_map={'Go': '#EF553B', 'All Languages': '#636EFA'},
)
fig.update_layout(xaxis_title='Year', yaxis_title='# CVEs', legend_title='Dataset')
fig.show()

---
## Level 2 — Vulnerability Characteristics (Go only)

In [ ]:
# ── CWE x Year heatmap — Go only ─────────────────────────────────────────────
# Mirror of eda.ipynb Cell 12
top_cwes_20 = df_exp['cwe'].value_counts().head(20).index

pivot_cwe_year = (
    df_exp[
        df_exp['cwe'].isin(top_cwes_20) & df_exp['year'].notna()
    ]
    .groupby(['cwe', 'year'])['cve_id']
    .nunique()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    pivot_cwe_year, annot=True, fmt='d',
    cmap='Blues', linewidths=0.4, ax=ax,
)
ax.set_title('CWE x Year — # CVEs (Go only)', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Year')
ax.set_ylabel('CWE')
plt.tight_layout()
plt.show()

In [ ]:
# ── Go CVEs by year (stacked by CWE top-5) ───────────────────────────────────
# Adapted from eda.ipynb Cell 13
top5_cwe = df_exp['cwe'].value_counts().head(5).index.tolist()
df_exp['cwe_group'] = df_exp['cwe'].apply(lambda c: c if c in top5_cwe else 'Other')

year_cwe = (
    df_exp.dropna(subset=['year'])
    .groupby(['year', 'cwe_group'])['cve_id']
    .nunique()
    .reset_index()
    .rename(columns={'cve_id': 'count'})
)

fig = px.bar(
    year_cwe, x='year', y='count',
    color='cwe_group',
    barmode='stack',
    title='Go CVEs by Year — stacked by CWE (top 5 + Other)',
    color_discrete_sequence=PALETTE,
)
fig.update_layout(xaxis_title='Year', yaxis_title='# CVEs', legend_title='CWE')
fig.show()

---
## Level 3 — Patch Complexity (Go only)

In [ ]:
# ── Descriptive stats — Mirror of eda.ipynb Cell 15 ──────────────────────────
print('Go subset patch metrics:')
go_stats = df[['patch_lines', 'patch_hunks', 'patch_files']].describe().round(2)

print('\nFull dataset patch metrics:')
all_stats = df_all[['patch_lines', 'patch_hunks', 'patch_files']].describe().round(2)

# Side-by-side
comparison = pd.concat([go_stats, all_stats], axis=1, keys=['Go', 'All'])
comparison

In [ ]:
# ── Distribution of patch metrics — Go vs Full — Mirror of eda.ipynb Cell 16 ──
metrics = ['patch_lines', 'patch_hunks', 'patch_files']

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Patch Lines', 'Patch Hunks', 'Patch Files'],
)

colors_go  = ['#EF553B', '#EF553B', '#EF553B']
colors_all = ['#636EFA', '#00CC96', '#AB63FA']

for col_idx, metric in enumerate(metrics, start=1):
    fig.add_trace(
        go.Histogram(x=df_all[metric], name='All', marker_color=colors_all[col_idx-1],
                     nbinsx=30, opacity=0.5, legendgroup='All',
                     showlegend=(col_idx == 1)),
        row=1, col=col_idx,
    )
    fig.add_trace(
        go.Histogram(x=df[metric], name='Go', marker_color=colors_go[col_idx-1],
                     nbinsx=30, opacity=0.8, legendgroup='Go',
                     showlegend=(col_idx == 1)),
        row=1, col=col_idx,
    )

fig.update_layout(
    title_text='Distribution of Patch Metrics — Go (red) vs All (blue/green/purple)',
    barmode='overlay', height=400,
    legend_title='Dataset',
)
fig.show()

In [ ]:
# ── Complexity tier — Go vs All — Mirror of eda.ipynb Cell 17 ─────────────────
tier_go  = df['complexity'].value_counts().reindex(COMPLEXITY_ORDER).reset_index()
tier_go.columns = ['complexity', 'count']
tier_go['source'] = 'Go'

tier_all = df_all['complexity'].value_counts().reindex(COMPLEXITY_ORDER).reset_index()
tier_all.columns = ['complexity', 'count']
tier_all['source'] = 'All Languages'

tier_combined = pd.concat([tier_all, tier_go])
tier_combined['pct'] = tier_combined.groupby('source')['count'].transform(
    lambda x: x / x.sum() * 100
).round(1)

fig = px.bar(
    tier_combined, x='complexity', y='pct',
    color='source', barmode='group',
    title='Patch Complexity Tier — Go vs All Languages (%)',
    category_orders={'complexity': COMPLEXITY_ORDER},
    color_discrete_map={'Go': '#EF553B', 'All Languages': '#636EFA'},
    text=tier_combined['pct'].apply(lambda v: f'{v}%'),
)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Complexity Tier', yaxis_title='% CVEs', legend_title='Dataset'
)
fig.show()

In [ ]:
# ── CWE x Patch Complexity — Go only — Mirror of eda.ipynb Cell 18 ────────────
pivot_cwe_complexity = (
    df_exp[df_exp['cwe'].isin(top_cwes_20)]
    .groupby(['cwe', 'complexity'])['cve_id']
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=COMPLEXITY_ORDER, fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    pivot_cwe_complexity, annot=True, fmt='d',
    cmap='RdYlGn_r', linewidths=0.5, ax=ax,
)
ax.set_title('CWE x Patch Complexity — Go only', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Patch Complexity')
ax.set_ylabel('CWE')
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter: patch_lines vs patch_hunks — Mirror of eda.ipynb Cell 22 ────────
fig = px.scatter(
    df, x='patch_lines', y='patch_hunks',
    size='patch_files',
    hover_name='cve_id',
    color='complexity',
    color_discrete_map=COMPLEXITY_COLORS,
    title='Patch Lines vs Hunks — Go (bubble size = # files changed)',
    opacity=0.8,
)
fig.update_layout(xaxis_title='Patch Lines', yaxis_title='Patch Hunks', legend_title='Complexity')
fig.show()

---
## Level 4 — PoC Results vs Baseline PatchEval

So sánh kết quả **gemini-3.5-flash** (tự chạy trên Go PoC) với **3 model baseline** của PatchEval paper.

> Baseline từ `eda.ipynb` Cell 29 — chạy trên 80 Go CVEs:
> 
> | Model | n_total | n_success | Rate |
> |---|---|---|---|
> | claude-3-5-sonnet | 80 | 66 | 82.5% |
> | gemini-1.5-pro | 80 | 72 | 90.0% |
> | gpt-4o | 80 | 72 | 90.0% |

In [ ]:
# ── Load PoC evaluation results from logs ─────────────────────────────────────
def parse_log_dir(log_dir):
    records = []
    for cve in os.listdir(log_dir):
        cve_dir   = Path(log_dir) / cve
        success_f = cve_dir / 'success_output.log'
        error_f   = cve_dir / 'error_output.log'
        if success_f.exists():
            content, is_success = success_f.read_text(errors='replace'), True
        elif error_f.exists():
            content, is_success = error_f.read_text(errors='replace'), False
        else:
            continue
        m = re.search(r'\[Validation TYPE\]: (.+)', content)
        vtype = m.group(1).strip() if m else 'unknown'
        records.append({'cve_id': cve, 'success': is_success, 'validation_type': vtype})
    return pd.DataFrame(records)

df_poc = parse_log_dir(LOG_DIR)

# Merge with Go dataset metadata (same fields as df_llm in eda.ipynb)
df_poc = df_poc.merge(
    df[['cve_id', 'cwe', 'patch_lines', 'patch_hunks', 'patch_files', 'complexity', 'year']],
    on='cve_id', how='left'
)

# Add model label from patch file
patches = {}
if PATCH_FILE.exists():
    with PATCH_FILE.open() as f:
        for line in f:
            if line.strip():
                p = json.loads(line)
                patches[p['cve']] = p.get('model', 'unknown')
df_poc['model'] = df_poc['cve_id'].map(patches).fillna('unknown')
# Treat df_poc as df_llm going forward
df_llm = df_poc.copy()

print(f'CVEs evaluated : {len(df_llm)}')
print(f'Models         : {df_llm["model"].unique().tolist()}')
print(f'Success        : {df_llm["success"].sum()} ({df_llm["success"].mean():.1%})')
df_llm.head(3)

In [ ]:
# ── Overall model performance — Mirror of eda.ipynb Cell 29 ──────────────────
# Baseline PatchEval numbers (from eda.ipynb cell 29 output)
baseline_rows = [
    {'model': 'claude-3-5-sonnet', 'success_rate': 0.825, 'n_success': 66, 'n_total': 80, 'source': 'PatchEval Baseline'},
    {'model': 'gemini-1.5-pro',    'success_rate': 0.900, 'n_success': 72, 'n_total': 80, 'source': 'PatchEval Baseline'},
    {'model': 'gpt-4o',            'success_rate': 0.900, 'n_success': 72, 'n_total': 80, 'source': 'PatchEval Baseline'},
]
df_baseline = pd.DataFrame(baseline_rows)

# Our run
overall_ours = (
    df_llm.groupby('model')['success']
    .agg(n_success='sum', n_total='count')
    .reset_index()
)
overall_ours['success_rate'] = overall_ours['n_success'] / overall_ours['n_total']
overall_ours['source'] = 'Our Run (Go PoC)'

df_compare = pd.concat([df_baseline, overall_ours[['model','success_rate','n_success','n_total','source']]], ignore_index=True)
df_compare['success_rate_pct'] = (df_compare['success_rate'] * 100).round(1)
df_compare

In [ ]:
# ── Bar chart: Our model vs Baseline — overall success rate ───────────────────
fig = px.bar(
    df_compare, x='model', y='success_rate',
    color='source',
    barmode='group',
    color_discrete_map={'PatchEval Baseline': '#636EFA', 'Our Run (Go PoC)': '#EF553B'},
    text=df_compare['success_rate_pct'].apply(lambda v: f'{v}%'),
    title='Overall Success Rate — Our Run vs PatchEval Baseline (Go CVEs)',
    hover_data=['n_total', 'n_success'],
)
fig.update_traces(textposition='outside')
avg_baseline = df_baseline['success_rate'].mean()
fig.add_hline(
    y=avg_baseline, line_dash='dot', line_color='gray',
    annotation_text=f'Avg Baseline: {avg_baseline:.1%}',
    annotation_position='top right',
)
fig.update_layout(
    yaxis_tickformat='.0%', yaxis_range=[0, 1.1],
    xaxis_title='Model', yaxis_title='Success Rate',
    legend_title='Source',
)
fig.show()

In [ ]:
# ── P(repair success | Patch Complexity) — Mirror of eda.ipynb Cell 26 ────────
# Our run
our_complexity = (
    df_llm.groupby('complexity')['success']
    .mean().reset_index().rename(columns={'success': 'success_rate'})
)
our_complexity['model'] = df_llm['model'].iloc[0]
our_complexity['source'] = 'Our Run'

# Baseline (reconstructed from eda.ipynb — approximated per complexity if not available)
# Using aggregate baseline success rates as flat reference lines
baseline_models = [
    ('claude-3-5-sonnet', 0.825),
    ('gemini-1.5-pro',    0.900),
    ('gpt-4o',            0.900),
]

our_complexity['complexity'] = pd.Categorical(
    our_complexity['complexity'], categories=COMPLEXITY_ORDER, ordered=True
)
our_complexity = our_complexity.sort_values('complexity')

fig = go.Figure()

# Our run line
fig.add_trace(go.Scatter(
    x=our_complexity['complexity'], y=our_complexity['success_rate'],
    mode='lines+markers', name=f'Our Run ({df_llm["model"].iloc[0]})',
    line=dict(color='#EF553B', width=3),
    marker=dict(size=10),
))

# Baseline flat reference lines
baseline_colors = ['#636EFA', '#00CC96', '#AB63FA']
for (model_name, rate), color in zip(baseline_models, baseline_colors):
    fig.add_hline(y=rate, line_dash='dash', line_color=color,
                  annotation_text=f'{model_name}: {rate:.0%}',
                  annotation_position='top left')

fig.update_layout(
    title='P(repair success | Patch Complexity) — Our Run vs Baseline',
    xaxis_title='Patch Complexity',
    yaxis_title='Success Rate',
    yaxis_tickformat='.0%',
    yaxis_range=[0, 1.05],
    legend_title='Model',
)
fig.show()

In [ ]:
# ── P(repair success | CWE) — Mirror of eda.ipynb Cell 27 ────────────────────
top_cwes_go = df_exp['cwe'].value_counts().head(10).index

success_cwe = (
    df_llm[df_llm['cwe'].isin(top_cwes_go)]
    .groupby('cwe')['success']
    .agg(success_rate='mean', n='count')
    .reset_index()
    .sort_values('success_rate')
)

fig = px.bar(
    success_cwe, x='success_rate', y='cwe',
    orientation='h',
    color='success_rate',
    color_continuous_scale='RdYlGn',
    text=success_cwe['success_rate'].apply(lambda v: f'{v:.1%}'),
    title='P(repair success | CWE) — Our Run, Top 10 Go CWEs',
    hover_data=['n'],
)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_tickformat='.0%', xaxis_title='Success Rate',
    yaxis_title='CWE', coloraxis_showscale=False,
    xaxis_range=[0, 1.1],
)
fig.show()

In [ ]:
# ── P(repair success | Single vs Multi-file) — Mirror of eda.ipynb Cell 28 ───
df_llm['multi_file'] = df_llm['patch_files'].apply(
    lambda n: 'Multi-file' if n > 1 else 'Single-file'
)

success_file = (
    df_llm.groupby('multi_file')['success']
    .agg(success_rate='mean', n='count')
    .reset_index()
)

fig = px.bar(
    success_file, x='multi_file', y='success_rate',
    color='multi_file',
    color_discrete_map={'Single-file': '#636EFA', 'Multi-file': '#EF553B'},
    text=success_file['success_rate'].apply(lambda v: f'{v:.1%}'),
    title='P(repair success | Single-file vs Multi-file) — Our Run (Go)',
    hover_data=['n'],
)
fig.update_traces(textposition='outside')
fig.update_layout(
    yaxis_tickformat='.0%', yaxis_range=[0, 1.1],
    xaxis_title='Patch Scope', yaxis_title='Success Rate',
    showlegend=False,
)
fig.show()

In [ ]:
# ── Failure type breakdown — additional vs eda.ipynb ─────────────────────────
vtype_counts = df_llm['validation_type'].value_counts().reset_index()
vtype_counts.columns = ['validation_type', 'count']

COLOR_MAP = {
    'Repair Success' : '#2ecc71',
    'validation_fail': '#e67e22',
    'compilation_fail': '#e74c3c',
    'apply_fail'     : '#c0392b',
    'unknown'        : '#95a5a6',
}

fig = px.bar(
    vtype_counts, x='validation_type', y='count',
    color='validation_type',
    color_discrete_map=COLOR_MAP,
    text='count',
    title='Failure Type Breakdown — Our Run (Go PoC)',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Validation Type', yaxis_title='# CVEs', showlegend=False
)
fig.show()

In [ ]:
# ── Final comparison table ── Mirror + extension of eda.ipynb Cell 29 ─────────
print('=' * 65)
print('FINAL COMPARISON — Our Run vs PatchEval Baseline (Go CVEs)')
print('=' * 65)
print(f'{"Model":<25} {"Source":<22} {"n_total":>8} {"n_success":>10} {"Rate":>7}')
print('-' * 65)
for _, row in df_compare.iterrows():
    marker = '  <-- OUR RUN' if row['source'] == 'Our Run (Go PoC)' else ''
    print(f"{row['model']:<25} {row['source']:<22} {int(row['n_total']):>8} {int(row['n_success']):>10} {row['success_rate']:>6.1%}  {marker}")

our_rate = overall_ours['success_rate'].iloc[0]
avg_base = df_baseline['success_rate'].mean()
diff     = our_rate - avg_base
print()
print(f'Average baseline : {avg_base:.1%}')
print(f'Our run          : {our_rate:.1%}')
print(f'Gap              : {diff:+.1%} ({"above" if diff >= 0 else "below"} baseline avg)')